### C95 pole Comparison based on Meert and Santosh (2022) https://doi.org/10.1016/j.gr.2022.06.014
### Bayesian pole analysis based on Heslop and Roberts (2019) https://doi.org/10.1029/2019JB018342

In [3]:
##Note: Meert and Santosh suggest warning when the Bayes error using C95 is < 0.2 

In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import combinations
from matplotlib.backends.backend_pdf import PdfPages

In [5]:
# ============================================================
# USER SETTINGS
# ============================================================

# Setup your own input/output file paths.

INPUT_CSV = r"c:\users\jmeer\poletest.csv"
OUTPUT_DIR = r"c:\users\jmeer\pole_pair_statistics"

#Column names in your file should match these or use your own.

COL_NAME = "name"
COL_LAT = "pole_lat"
COL_LON = "pole_lon"
COL_A95 = "a95"
COL_AGE = "age"
COL_AGEERR = "age_error"
COL_N = "N"

N_PLOT_POINTS = 1200

#Note: This rate is based on the analysis in Torsvik et al., 2012 and can be changed if desired)

APW_RATE_DEG_PER_MA = 0.44

# ============================================================
# BASIC FUNCTIONS
# ============================================================

def sph_to_cart(lat_deg, lon_deg):
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)

    return np.array([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ], dtype=float)


def angular_distance(lat1, lon1, lat2, lon2):
    v1 = sph_to_cart(lat1, lon1)
    v2 = sph_to_cart(lat2, lon2)

    return np.degrees(
        np.arccos(np.clip(np.dot(v1, v2), -1.0, 1.0))
    )


# ============================================================
# FISHER / C95 FUNCTIONS
# ============================================================

def pairwise_C95(a95_1, a95_2, age1, age2):
    """
    Pairwise C95 including temporal correction:

    C95 = sqrt(A95_1^2 + A95_2^2 + (0.44 * ΔT)^2)
    """

    age_diff = abs(age1 - age2)
    age_term = APW_RATE_DEG_PER_MA * age_diff

    return np.sqrt(a95_1**2 + a95_2**2 + age_term**2)


def fisher_R_K_from_alpha95(alpha95_deg, N):
    """
    Fisher confidence-cone relationship.

    cos(alpha95) = 1 - ((N - R) / R) *
                   (20^(1/(N-1)) - 1)

    R = N / [1 + (1 - cos(alpha95)) / F]

    F = 20^(1/(N-1)) - 1

    K = (N - 1) / (N - R)
    """

    if N <= 1:
        raise ValueError("N must be greater than 1.")

    alpha = np.radians(alpha95_deg)
    F = 20.0 ** (1.0 / (N - 1.0)) - 1.0

    R = N / (1.0 + (1.0 - np.cos(alpha)) / F)

    if R >= N:
        R = N - 1e-12

    K = (N - 1.0) / (N - R)

    return R, K


# ============================================================
# SIMILARITY FUNCTIONS
# ============================================================

def bhattacharyya_coefficient_vmf(mu1, kappa1, mu2, kappa2):
    """
    Analytic Bhattacharyya coefficient for two Fisher/vMF distributions.
    """

    q = 0.5 * (kappa1 * mu1 + kappa2 * mu2)
    qnorm = np.linalg.norm(q)

    def log_sinh(x):
        if x > 700:
            return x - np.log(2.0)
        return np.log(np.sinh(x))

    if qnorm < 1e-12:
        log_sinh_term = 0.0
    else:
        log_sinh_term = log_sinh(qnorm) - np.log(qnorm)

    log_bc = (
        0.5 * (np.log(kappa1) + np.log(kappa2))
        - 0.5 * (log_sinh(kappa1) + log_sinh(kappa2))
        + log_sinh_term
    )

    return float(np.exp(log_bc))


def bayes_error_from_bhattacharyya(B):
    """
    Bayes error from Bhattacharyya coefficient.

    Benchmark:
    B = 0.12 gives Bayes error ≈ 0.06.
    """

    B = np.clip(B, 0.0, 1.0)
    return 0.5 * B


def fisher_profile(theta_deg, kappa):
    """
    Relative 1-D Fisher/vMF profile for plotting only.
    """

    theta = np.radians(theta_deg)
    logp = kappa * np.cos(theta)
    logp -= np.nanmax(logp)

    return np.exp(logp)


# ============================================================
# MAIN Analysis
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(INPUT_CSV)

required = [
    COL_NAME,
    COL_LAT,
    COL_LON,
    COL_A95,
    COL_AGE,
    COL_AGEERR,
    COL_N
]

for col in required:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

df = df.dropna(subset=required).reset_index(drop=True)

rows = []

csv_out = os.path.join(
    OUTPUT_DIR,
    "pairwise_C95_Bhatt_Bayes_final.csv"
)

pdf_out = os.path.join(
    OUTPUT_DIR,
    "pairwise_C95_Bhatt_Bayes_final_PDFs.pdf"
)

with PdfPages(pdf_out) as pdf:

    for i, j in combinations(range(len(df)), 2):

        p1 = df.loc[i]
        p2 = df.loc[j]

        name1 = str(p1[COL_NAME])
        name2 = str(p2[COL_NAME])

        lat1 = float(p1[COL_LAT])
        lon1 = float(p1[COL_LON])
        a95_1 = float(p1[COL_A95])
        age1 = float(p1[COL_AGE])
        ageerr1 = float(p1[COL_AGEERR])
        N1 = int(p1[COL_N])

        lat2 = float(p2[COL_LAT])
        lon2 = float(p2[COL_LON])
        a95_2 = float(p2[COL_A95])
        age2 = float(p2[COL_AGE])
        ageerr2 = float(p2[COL_AGEERR])
        N2 = int(p2[COL_N])

        if N1 <= 1 or N2 <= 1:
            print(f"Skipping {name1} vs {name2}: N must be > 1")
            continue

        mu1 = sph_to_cart(lat1, lon1)
        mu2 = sph_to_cart(lat2, lon2)

        sep = angular_distance(lat1, lon1, lat2, lon2)

        age_diff = abs(age1 - age2)
        age_term = APW_RATE_DEG_PER_MA * age_diff

        # ====================================================
        # ORIGINAL A95-BASED PARAMETERS
        # ====================================================

        R1_A95, K1_A95 = fisher_R_K_from_alpha95(a95_1, N1)
        R2_A95, K2_A95 = fisher_R_K_from_alpha95(a95_2, N2)

        kappa1_A95 = K1_A95 * R1_A95
        kappa2_A95 = K2_A95 * R2_A95

        bhatt_A95 = bhattacharyya_coefficient_vmf(
            mu1, kappa1_A95,
            mu2, kappa2_A95
        )

        bayes_A95 = bayes_error_from_bhattacharyya(bhatt_A95)

        # ====================================================
        # C95-BASED PARAMETERS
        # ====================================================

        C95 = pairwise_C95(a95_1, a95_2, age1, age2)

        R1_C95, K1_C95 = fisher_R_K_from_alpha95(C95, N1)
        R2_C95, K2_C95 = fisher_R_K_from_alpha95(C95, N2)

        kappa1_C95 = K1_C95 * R1_C95
        kappa2_C95 = K2_C95 * R2_C95

        bhatt_C95 = bhattacharyya_coefficient_vmf(
            mu1, kappa1_C95,
            mu2, kappa2_C95
        )

        bayes_C95 = bayes_error_from_bhattacharyya(bhatt_C95)

        sep_le_C95 = sep <= C95
        sep_over_C95 = sep / C95 if C95 > 0 else np.nan

        # ====================================================
        # PDF GRAPHIC USING C95-DERIVED EFFECTIVE KAPPA
        # ====================================================

        xpad = max(C95 * 2.5, sep * 0.25, 5.0)
        x = np.linspace(-xpad, sep + xpad, N_PLOT_POINTS)

        pdf1 = fisher_profile(np.abs(x), kappa1_C95)
        pdf2 = fisher_profile(np.abs(x - sep), kappa2_C95)

        fig, ax = plt.subplots(figsize=(8.8, 5.3))

        ax.plot(
            x,
            pdf1,
            linewidth=2.0,
            label=f"{name1}: κ={kappa1_C95:.2f}, K={K1_C95:.2f}, R={R1_C95:.3f}, N={N1}"
        )

        ax.plot(
            x,
            pdf2,
            linewidth=2.0,
            label=f"{name2}: κ={kappa2_C95:.2f}, K={K2_C95:.2f}, R={R2_C95:.3f}, N={N2}"
        )

        ax.fill_between(
            x,
            np.minimum(pdf1, pdf2),
            color="0.75",
            alpha=0.65,
            label="PDF overlap"
        )

        ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
        ax.axvline(sep, color="black", linestyle="--", linewidth=0.8)

        ax.axvspan(
            -C95,
            C95,
            color="blue",
            alpha=0.10,
            label=f"{name1} C95"
        )

        ax.axvspan(
            sep - C95,
            sep + C95,
            color="red",
            alpha=0.10,
            label=f"{name2} C95"
        )

        ax.set_xlabel("Angular distance along pole-pair great circle (degrees)")
        ax.set_ylabel("Relative probability density")
        ax.set_title(f"{name1} vs {name2}")

        ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)
        ax.legend(fontsize=6.8)

# Note Bhatt A95 comparisons are based on raw data without any age comparisons

        ax.text(
            0.03,
            0.95,
            (
                f"Pole separation = {sep:.2f}°\n"
                f"A95-1 = {a95_1:.2f}°, N1 = {N1}\n"
                f"A95-2 = {a95_2:.2f}°, N2 = {N2}\n"
                f"ΔT = {age_diff:.2f} Ma\n"
                f"0.44 × ΔT = {age_term:.2f}°\n"
                f"C95 = {C95:.2f}°\n"
                f"R1(C95) = {R1_C95:.5f}, K1(C95) = {K1_C95:.3f}, κ1 = {kappa1_C95:.3f}\n"
                f"R2(C95) = {R2_C95:.5f}, K2(C95) = {K2_C95:.3f}, κ2 = {kappa2_C95:.3f}\n"
                f"Separation/C95 = {sep_over_C95:.3f}\n"
                f"Separation ≤ C95 = {sep_le_C95}\n"
                f"Bhatt C95 = {bhatt_C95:.4f}\n"
                f"Bayes C95 = {bayes_C95:.4f}\n"
                f"Bhatt A95 = {bhatt_A95:.4f}\n"
                f"Bayes A95 = {bayes_A95:.4f}"
            ),
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=7.5,
            bbox=dict(facecolor="white", edgecolor="0.5", alpha=0.88)
        )

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        rows.append({
            "pole_1": name1,
            "pole_2": name2,

            "age_1_Ma": age1,
            "age_2_Ma": age2,
            "age_error_1_Ma": ageerr1,
            "age_error_2_Ma": ageerr2,
            "age_difference_Ma": age_diff,
            "age_term_0p44_deg_per_Ma": age_term,

            "pole_lat_1": lat1,
            "pole_lon_1": lon1,
            "pole_lat_2": lat2,
            "pole_lon_2": lon2,
            "angular_separation_deg": sep,

            "N_1": N1,
            "N_2": N2,

            "A95_1": a95_1,
            "A95_2": a95_2,

            "R_1_from_A95": R1_A95,
            "K_1_from_A95": K1_A95,
            "kappa_1_A95_K_times_R": kappa1_A95,

            "R_2_from_A95": R2_A95,
            "K_2_from_A95": K2_A95,
            "kappa_2_A95_K_times_R": kappa2_A95,

            "Bhattacharyya_A95": bhatt_A95,
            "Bayes_error_A95": bayes_A95,

            "pairwise_C95_with_temporal_term": C95,

            "R_1_from_C95": R1_C95,
            "K_1_from_C95": K1_C95,
            "kappa_1_C95_K_times_R": kappa1_C95,

            "R_2_from_C95": R2_C95,
            "K_2_from_C95": K2_C95,
            "kappa_2_C95_K_times_R": kappa2_C95,

            "separation_le_C95": sep_le_C95,
            "separation_over_C95": sep_over_C95,

            "Bhattacharyya_C95": bhatt_C95,
            "Bayes_error_C95": bayes_C95
        })

# ============================================================
# SAVE CSV You can change as needed
# ============================================================

out = pd.DataFrame(rows)

out = out.sort_values(
    by="Bayes_error_C95",
    ascending=False
).reset_index(drop=True)

out.to_csv(csv_out, index=False)

print("Done.")
print("CSV table:", csv_out)
print("PDF graphics:", pdf_out)

Done.
CSV table: c:\users\jmeer\pole_pair_statistics\pairwise_C95_Bhatt_Bayes_final.csv
PDF graphics: c:\users\jmeer\pole_pair_statistics\pairwise_C95_Bhatt_Bayes_final_PDFs.pdf
